# Criando camada gold (Adicionando Views e Otimizações) 🥇

In [0]:
catalogo = "medalhao_at"
schema_silver = "silver"
schema_gold = "gold"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window 

## 💼 📊 Visão Comercial e Volume de Produtos

### Criando Gold 

In [0]:
# Salvando as tabelas necessárias pra criar a nossa gold 
tb_pedido_total = spark.table(f"{catalogo}.{schema_silver}.fat_pedido_total")
tb_itens_pedidos = spark.table(f"{catalogo}.{schema_silver}.fat_itens_pedidos_validos")
tb_produtos = spark.table(f"{catalogo}.{schema_silver}.dim_produtos_validos")

# Verificando os dados dessas tabelas pra confirmação da escolha das tabelas a serem usadas 
display(tb_pedido_total.limit(10))
display(tb_itens_pedidos.limit(10))
display(tb_produtos.limit(10))

id_pedido,id_consumidor,status,valor_total_pago_brl,valor_total_pago_usd,data_pedido
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,entregue,38.71,12.24,2017-10-02T10:56:33.000Z
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,entregue,141.46,37.77,2018-07-24T20:41:37.000Z
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,entregue,179.12,47.75,2018-08-08T08:38:49.000Z
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,entregue,72.2,22.02,2017-11-18T19:28:06.000Z
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,entregue,28.62,8.72,2018-02-13T21:18:39.000Z
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,entregue,175.26,53.29,2017-07-09T21:57:05.000Z
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,faturado,65.95,20.99,2017-04-11T12:22:08.000Z
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,entregue,75.16,24.31,2017-05-16T13:10:30.000Z
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,entregue,35.95,11.38,2017-01-23T18:29:09.000Z
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,entregue,169.76,53.98,2017-07-29T11:55:02.000Z


id_pedido,id_item_pedido,id_produto,id_vendedor,data_entrega_limite,preco,frete,data_ingestao,flag_id_pedido_valido,flag_id_pedido_item_valido,flag_produto_valido,flag_vendedor_valido,flag_preco_valido,flag_frete_valido,flag_qualidade
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19T09:45:35.000Z,58.9,13.29,2026-04-05T10:46:57.892Z,true,true,true,true,true,true,OK
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03T11:05:13.000Z,239.9,19.93,2026-04-05T10:46:57.892Z,true,true,true,true,true,true,OK
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18T14:48:30.000Z,199.0,17.87,2026-04-05T10:46:57.892Z,true,true,true,true,true,true,OK
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15T10:10:18.000Z,12.99,12.79,2026-04-05T10:46:57.892Z,true,true,true,true,true,true,OK
00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13T13:57:51.000Z,199.9,18.14,2026-04-05T10:46:57.892Z,true,true,true,true,true,true,OK
00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,2017-05-23T03:55:27.000Z,21.9,12.69,2026-04-05T10:46:57.892Z,true,true,true,true,true,true,OK
00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2017-12-14T12:10:31.000Z,19.9,11.85,2026-04-05T10:46:57.892Z,true,true,true,true,true,true,OK
000576fe39319847cbb9d288c5617fa6,1,557d850972a7d6f792fd18ae1400d9b6,5996cddab893a4652a15592fb58ab8db,2018-07-10T12:30:45.000Z,810.0,70.75,2026-04-05T10:46:57.892Z,true,true,true,true,true,true,OK
0005a1a1728c9d785b8e2b08b904576c,1,310ae3c140ff94b03219ad0adc3c778f,a416b6a846a11724393025641d4edd5e,2018-03-26T18:31:29.000Z,145.95,11.65,2026-04-05T10:46:57.892Z,true,true,true,true,true,true,OK
0005f50442cb953dcd1d21e1fb923495,1,4535b0e1091c278dfd193e5a1d63b39f,ba143b05f0110f0dc71ad71b4466ce92,2018-07-06T14:10:56.000Z,53.99,11.4,2026-04-05T10:46:57.892Z,true,true,true,true,true,true,OK


id_produto,nome_produto,categoria_produto,tamanho_nome_produto,tamanho_descricao_produto,quantidade_fotos,peso_produto_gramas,comprimento_centimetros,altura_centimetros,largura_centimetros,data_ingestao,flag_id_valido,flag_nome_valido,flag_qtd_fotos_valida,flag_qualidade
00066f42aeeb9f3007548bb9d3f33c38,Loção Corporal Preto,PERFUMARIA,53,596,6,300,20,16,16,2026-04-05T10:47:13.723Z,true,true,true,OK
00088930e925c41fd95ebfe695fd2655,Central Multimídia Avançado,AUTOMOTIVO,56,752,4,1225,55,10,26,2026-04-05T10:47:13.723Z,true,true,true,OK
0009406fd7479715e4bef61dd91f2462,Toalha De Banho Premium,CAMA_MESA_BANHO,50,266,2,300,45,15,35,2026-04-05T10:47:13.723Z,true,true,true,OK
000b8f95fcb9e0096488278317764d19,Tábua De Corte,UTILIDADES_DOMESTICAS,25,364,3,550,19,24,12,2026-04-05T10:47:13.723Z,true,true,true,OK
000d9be29b5207b54e86aa1b1ac54872,Relógio Analógico Dourado,RELOGIOS_PRESENTES,48,613,4,250,22,11,15,2026-04-05T10:47:13.723Z,true,true,true,OK
0011c512eb256aa0dbbb544d8dffcf6e,Bateria Automotiva Luxo,AUTOMOTIVO,58,177,1,100,16,15,16,2026-04-05T10:47:13.723Z,true,true,true,OK
00126f27c813603687e6ce486d909d01,Quadro Nerd Master,COOL_STUFF,42,2461,1,700,25,5,15,2026-04-05T10:47:13.723Z,true,true,true,OK
001795ec6f1b187d37335e1c4704762e,Acessório Padrão Luxo,CONSOLES_GAMES,53,274,1,600,30,20,20,2026-04-05T10:47:13.723Z,true,true,true,OK
001b237c0e9bb435f2e54071129237e9,Edredom,CAMA_MESA_BANHO,42,253,1,6000,40,4,30,2026-04-05T10:47:13.723Z,true,true,true,OK
001b72dfd63e9833e8c02742adf472e3,Mesa De Jantar,MOVEIS_DECORACAO,45,520,3,600,26,8,22,2026-04-05T10:47:13.723Z,true,true,true,OK


In [0]:
# 1. Leitura das Tabelas Silver
# Foi feita releitura pra facilitar o criação das tabelas pro pipeline 
# Dessa forma, não é necessário rodar o notebook anterior sempre
df_pedidos = spark.table(f"{catalogo}.{schema_silver}.fat_pedido_total")
df_itens = spark.table(f"{catalogo}.{schema_silver}.fat_itens_pedidos_validos")
df_produtos = spark.table(f"{catalogo}.{schema_silver}.dim_produtos_validos")

# 2. Transformações Iniciais
# Extraindo o Ano e o Mês da data do pedido 
df_pedidos = df_pedidos.withColumn("ano_venda", F.year("data_pedido")) \
                       .withColumn("mes_venda", F.month("data_pedido"))

# 3. Join das tabelas
# Obs: A ordem das tabelas importa, e portanto, como tabela central foi escolhida a fat_itens_pedidos_validos
# Essa tabela foi escolhida pois apenas ela possuia tanto o id_produto necessário pra construção dessa gold, como também o id_pedido, para outros cálculos necessários 
df_join = (
    df_itens
    .join(df_pedidos, on="id_pedido", how="inner")
    .join(df_produtos, on="id_produto", how="inner")
)

# 4. Agrupamento e agregação
# O agg é a abreviação pra função de agregação 
# É isso que permite adicionar colunas advindas de cálculos, seja ele um count ou outra coisa mais complexa
df_agrupado = df_join.groupBy("ano_venda", "mes_venda", "categoria_produto").agg(
    # Contagem da quantidade de pedidos distintos
    F.countDistinct("id_pedido").alias("total_pedidos"),

    # Contagem da quantidade absoluta de itens vendidos
    F.count("id_item_pedido").alias("qtd_itens_vendidos"),
    
    # Soma dos valores dos produtos vendidos 
    F.sum("preco").alias("receita_total_brl_bruta"),
    
    # Para achar o USD do item proporcional, aplicamos a taxa de conversão do pedido àquele item
    F.sum(
        F.when(F.col("valor_total_pago_brl") > 0, 
               F.col("preco") * (F.col("valor_total_pago_usd") / F.col("valor_total_pago_brl")))
         .otherwise(0)
    ).alias("receita_total_usd_bruta")
)

# 5. Cálculos adicionais
# Nessa etapa foram adicionados alguns cálculos a fim de facilitar a análise das informações necessárias 
df_vendas_comercial = df_agrupado.withColumn(
    "ticket_medio_brl", 
    F.col("receita_total_brl_bruta") / F.col("total_pedidos")
).select(
    "ano_venda",
    "mes_venda",
    "categoria_produto",
    "total_pedidos",
    "qtd_itens_vendidos",
    F.round("receita_total_brl_bruta", 2).alias("receita_total_brl"),
    F.round("receita_total_usd_bruta", 2).alias("receita_total_usd"),
    F.round("ticket_medio_brl", 2).alias("ticket_medio_brl")
)

# 6. Analisando se o dataframe possui as informações que deveria
df_vendas_comercial.display()

# 7. Escrevendo a tabela na camada gold 
df_vendas_comercial.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{schema_gold}.fat_vendas_comercial")

ano_venda,mes_venda,categoria_produto,total_pedidos,qtd_itens_vendidos,receita_total_brl,receita_total_usd,ticket_medio_brl
2017,8,CONSTRUCAO_FERRAMENTAS_CONSTRUCAO,6,7,1180.0,373.24,196.67
2018,8,MOVEIS_DECORACAO,344,433,41380.15,10787.09,120.29
2017,11,FASHION_BOLSAS_E_ACESSORIOS,185,200,13414.89,4127.66,72.51
2018,3,LIVROS_TECNICOS,17,17,913.47,279.25,53.73
2018,5,CAMA_MESA_BANHO,606,747,71702.77,19776.68,118.32
2018,1,BELEZA_SAUDE,572,628,72470.49,22592.9,126.7
2017,12,MOVEIS_COZINHA_AREA_DE_SERVICO_JANTAR_E_JARDIM,16,20,4277.3,1300.62,267.33
2018,6,AUTOMOTIVO,272,300,45407.49,12048.43,166.94
2017,8,COOL_STUFF,241,264,41720.03,13239.55,173.11
2017,3,AUTOMOTIVO,76,87,14482.07,4633.78,190.55


### Exibição via Display()

In [0]:
# 1. Junção da tabelas de produtos com a tabela de itens pra obtermos a informação que buscamos
df_produtos_vendidos = df_itens.join(
    df_produtos, 
    on="id_produto", 
    how="inner"
)

# 2. Calculando o total vendido por cada produto
df_ranking_produtos = df_produtos_vendidos.groupBy(
    "nome_produto", 
    "categoria_produto"
).agg(
    F.count("*").alias("quantidade_vendida")
)

# 3. Top 5 MAIS Vendidos (Ordem Decrescente)
df_top_5_mais_vendidos = (
    df_ranking_produtos
    .orderBy(F.col("quantidade_vendida").desc())
    .select(
        "nome_produto", 
        "categoria_produto", 
        "quantidade_vendida"
    )
    .limit(5)
)

# 4. Top 5 MENOS Vendidos (Ordem Crescente)
df_top_5_menos_vendidos = (
    df_ranking_produtos
    .orderBy(F.col("quantidade_vendida").asc())
    .select(
        "nome_produto", 
        "categoria_produto", 
        "quantidade_vendida"
    )
    .limit(5)
)

# ==============================================================================
# Exibição
# ==============================================================================
print("🏆 TOP 5 PRODUTOS MAIS VENDIDOS:")
df_top_5_mais_vendidos.display()

print("⚠️ TOP 5 PRODUTOS MENOS VENDIDOS:")
df_top_5_menos_vendidos.display()

🏆 TOP 5 PRODUTOS MAIS VENDIDOS:


nome_produto,categoria_produto,quantidade_vendida
Secador De Cabelo,BELEZA_SAUDE,1076
Toalha De Banho,CAMA_MESA_BANHO,919
Protetor Solar,BELEZA_SAUDE,917
Travesseiro,CAMA_MESA_BANHO,839
Colar,RELOGIOS_PRESENTES,804


⚠️ TOP 5 PRODUTOS MENOS VENDIDOS:


nome_produto,categoria_produto,quantidade_vendida
Acessório Padrão Max,FASHION_CALCADOS,1
Kit Variado Luxo,FASHION_ROUPA_MASCULINA,1
Máquina De Lavar Pro,ELETRODOMESTICOS,1
Peça De Reposição Branco,ELETROPORTATEIS,1
Equipamento Utilitário Básico,CASA_CONSTRUCAO,1


## 😁💬 Satisfação de Clientes e Qualidade de Parceiros

### Criando Gold

In [0]:
# Salvando as tabelas necessárias pra criar a nossa gold 
tb_avaliacoes = spark.table(f"{catalogo}.{schema_silver}.fat_avaliacoes_pedidos_validos")
tb_itens_pedidos = spark.table(f"{catalogo}.{schema_silver}.fat_itens_pedidos_validos")
tb_produtos = spark.table(f"{catalogo}.{schema_silver}.dim_produtos_validos")
tb_vendedores = spark.table(f"{catalogo}.{schema_silver}.dim_vendedores_validos")

# Verificando os dados dessas tabelas pra confirmação da escolha das tabelas a serem usadas 
print("Tabela fat_avaliacoes_pedidos_validos")
display(tb_avaliacoes.limit(10))

print("Tabela fat_itens_pedidos_validos")
display(tb_itens_pedidos.limit(10))

print("Tabela dim_produtos_validos")
display(tb_produtos.limit(10))

print("Tabela dim_vendedores_validos")
display(tb_vendedores.limit(10))

# Realizando count nas tabelas pra validação também 
print(f"Qtd linhas em tb_avaliacoes: {tb_avaliacoes.count()}")
print(f"Qtd linhas em tb_itens_pedidos: {tb_itens_pedidos.count()}")
print(f"Qtd linhas em tb_produtos: {tb_produtos.count()}")
print(f"Qtd linhas em tb_vendedores: {tb_vendedores.count()}")

Tabela fat_avaliacoes_pedidos_validos


id_avaliacao,id_pedido,nota_avaliacao,titulo_comentario_avaliacao,mensagem_comentario_avaliacao,data_criacao_comentario,data_resposta_comentario,data_ingestao,flag_id_avaliacao_valido,flag_id_pedido_valido,flag_nota_avaliacao_valido,flag_data_criacao_valida,flag_data_resposta_valida,flag_qualidade
7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,Sem título,Sem comentário,2018-01-18T00:00:00.000Z,2018-01-18T21:46:59.000Z,2026-04-05T10:47:05.637Z,true,true,true,true,true,OK
80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,Sem título,Sem comentário,2018-03-10T00:00:00.000Z,2018-03-11T03:05:13.000Z,2026-04-05T10:47:05.637Z,true,true,true,true,true,OK
228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,Sem título,Sem comentário,2018-02-17T00:00:00.000Z,2018-02-18T14:36:24.000Z,2026-04-05T10:47:05.637Z,true,true,true,true,true,OK
e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,Sem título,Recebi bem antes do prazo estipulado.,2017-04-21T00:00:00.000Z,2017-04-21T22:02:06.000Z,2026-04-05T10:47:05.637Z,true,true,true,true,true,OK
f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,Sem título,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa,2018-03-01T00:00:00.000Z,2018-03-02T10:26:53.000Z,2026-04-05T10:47:05.637Z,true,true,true,true,true,OK
15197aa66ff4d0650b5434f1b46cda19,b18dcdf73be66366873cd26c5724d1dc,1,Sem título,Sem comentário,2018-04-13T00:00:00.000Z,2018-04-16T00:39:37.000Z,2026-04-05T10:47:05.637Z,true,true,true,true,true,OK
07f9bee5d1b850860defd761afa7ff16,e48aa0d2dcec3a2e87348811bcfdf22b,5,Sem título,Sem comentário,2017-07-16T00:00:00.000Z,2017-07-18T19:30:34.000Z,2026-04-05T10:47:05.637Z,true,true,true,true,true,OK
7c6400515c67679fbee952a7525281ef,c31a859e34e3adac22f376954e19b39d,5,Sem título,Sem comentário,2018-08-14T00:00:00.000Z,2018-08-14T21:36:06.000Z,2026-04-05T10:47:05.637Z,true,true,true,true,true,OK
a3f6f7f6f433de0aefbb97da197c554c,9c214ac970e84273583ab523dfafd09b,5,Sem título,Sem comentário,2017-05-17T00:00:00.000Z,2017-05-18T12:05:37.000Z,2026-04-05T10:47:05.637Z,true,true,true,true,true,OK
8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,recomendo,aparelho eficiente. no site a marca do aparelho esta impresso como 3desinfector e ao chegar esta com outro nome...atualizar com a marca correta uma vez que é o mesmo aparelho,2018-05-22T00:00:00.000Z,2018-05-23T16:45:47.000Z,2026-04-05T10:47:05.637Z,true,true,true,true,true,OK


Tabela fat_itens_pedidos_validos


id_pedido,id_item_pedido,id_produto,id_vendedor,data_entrega_limite,preco,frete,data_ingestao,flag_id_pedido_valido,flag_id_pedido_item_valido,flag_produto_valido,flag_vendedor_valido,flag_preco_valido,flag_frete_valido,flag_qualidade
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19T09:45:35.000Z,58.9,13.29,2026-04-05T10:46:57.892Z,true,true,true,true,true,true,OK
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03T11:05:13.000Z,239.9,19.93,2026-04-05T10:46:57.892Z,true,true,true,true,true,true,OK
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18T14:48:30.000Z,199.0,17.87,2026-04-05T10:46:57.892Z,true,true,true,true,true,true,OK
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15T10:10:18.000Z,12.99,12.79,2026-04-05T10:46:57.892Z,true,true,true,true,true,true,OK
00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13T13:57:51.000Z,199.9,18.14,2026-04-05T10:46:57.892Z,true,true,true,true,true,true,OK
00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,2017-05-23T03:55:27.000Z,21.9,12.69,2026-04-05T10:46:57.892Z,true,true,true,true,true,true,OK
00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2017-12-14T12:10:31.000Z,19.9,11.85,2026-04-05T10:46:57.892Z,true,true,true,true,true,true,OK
000576fe39319847cbb9d288c5617fa6,1,557d850972a7d6f792fd18ae1400d9b6,5996cddab893a4652a15592fb58ab8db,2018-07-10T12:30:45.000Z,810.0,70.75,2026-04-05T10:46:57.892Z,true,true,true,true,true,true,OK
0005a1a1728c9d785b8e2b08b904576c,1,310ae3c140ff94b03219ad0adc3c778f,a416b6a846a11724393025641d4edd5e,2018-03-26T18:31:29.000Z,145.95,11.65,2026-04-05T10:46:57.892Z,true,true,true,true,true,true,OK
0005f50442cb953dcd1d21e1fb923495,1,4535b0e1091c278dfd193e5a1d63b39f,ba143b05f0110f0dc71ad71b4466ce92,2018-07-06T14:10:56.000Z,53.99,11.4,2026-04-05T10:46:57.892Z,true,true,true,true,true,true,OK


Tabela dim_produtos_validos


id_produto,nome_produto,categoria_produto,tamanho_nome_produto,tamanho_descricao_produto,quantidade_fotos,peso_produto_gramas,comprimento_centimetros,altura_centimetros,largura_centimetros,data_ingestao,flag_id_valido,flag_nome_valido,flag_qtd_fotos_valida,flag_qualidade
00066f42aeeb9f3007548bb9d3f33c38,Loção Corporal Preto,PERFUMARIA,53,596,6,300,20,16,16,2026-04-05T10:47:13.723Z,true,true,true,OK
00088930e925c41fd95ebfe695fd2655,Central Multimídia Avançado,AUTOMOTIVO,56,752,4,1225,55,10,26,2026-04-05T10:47:13.723Z,true,true,true,OK
0009406fd7479715e4bef61dd91f2462,Toalha De Banho Premium,CAMA_MESA_BANHO,50,266,2,300,45,15,35,2026-04-05T10:47:13.723Z,true,true,true,OK
000b8f95fcb9e0096488278317764d19,Tábua De Corte,UTILIDADES_DOMESTICAS,25,364,3,550,19,24,12,2026-04-05T10:47:13.723Z,true,true,true,OK
000d9be29b5207b54e86aa1b1ac54872,Relógio Analógico Dourado,RELOGIOS_PRESENTES,48,613,4,250,22,11,15,2026-04-05T10:47:13.723Z,true,true,true,OK
0011c512eb256aa0dbbb544d8dffcf6e,Bateria Automotiva Luxo,AUTOMOTIVO,58,177,1,100,16,15,16,2026-04-05T10:47:13.723Z,true,true,true,OK
00126f27c813603687e6ce486d909d01,Quadro Nerd Master,COOL_STUFF,42,2461,1,700,25,5,15,2026-04-05T10:47:13.723Z,true,true,true,OK
001795ec6f1b187d37335e1c4704762e,Acessório Padrão Luxo,CONSOLES_GAMES,53,274,1,600,30,20,20,2026-04-05T10:47:13.723Z,true,true,true,OK
001b237c0e9bb435f2e54071129237e9,Edredom,CAMA_MESA_BANHO,42,253,1,6000,40,4,30,2026-04-05T10:47:13.723Z,true,true,true,OK
001b72dfd63e9833e8c02742adf472e3,Mesa De Jantar,MOVEIS_DECORACAO,45,520,3,600,26,8,22,2026-04-05T10:47:13.723Z,true,true,true,OK


Tabela dim_vendedores_validos


id_vendedor,prefixo_cep,cidade,estado,nome_vendedor,data_registro_vendedor,data_ingestao,flag_id_vendedor,flag_cep_valido,flag_nome_vendedor_valido,flag_reistration_date_valido,flag_qualidade
0015a82c2db000af6aaaf3ae2ecb0532,9080,SANTO ANDRE,SP,Amanda Sá,2018-09-29,2026-04-05T10:47:17.095Z,true,true,true,true,OK
001cca7ae9ae17fb1caed9dfb1094831,29156,CARIACICA,ES,Vinicius Nogueira,2017-11-09,2026-04-05T10:47:17.095Z,true,true,true,true,OK
001e6ad469a905060d959994f1b41e4f,24754,SAO GONCALO,RJ,Ana Clara Moreira,2018-11-16,2026-04-05T10:47:17.095Z,true,true,true,true,OK
002100f778ceb8431b7a1020ff7ab48f,14405,FRANCA,SP,Srta. Emanuella Rezende,2017-04-17,2026-04-05T10:47:17.095Z,true,true,true,true,OK
003554e2dce176b5555353e4f3555ac8,74565,GOIANIA,GO,Rebeca Costa,2017-07-20,2026-04-05T10:47:17.095Z,true,true,true,true,OK
004c9cd9d87a3c30c522c48c4fc07416,14940,IBITINGA,SP,Aylla Da Rocha,2017-05-22,2026-04-05T10:47:17.095Z,true,true,true,true,OK
00720abe85ba0859807595bbf045a33b,7070,GUARULHOS,SP,Bruna Barbosa,2017-07-25,2026-04-05T10:47:17.095Z,true,true,true,true,OK
00ab3eff1b5192e5f1a63bcecfee11c8,4164,SAO PAULO,SP,Bella Da Luz,2017-11-18,2026-04-05T10:47:17.095Z,true,true,true,true,OK
00d8b143d12632bad99c0ad66ad52825,30170,BELO HORIZONTE,MG,Luiz Miguel Moura,2018-11-19,2026-04-05T10:47:17.095Z,true,true,true,true,OK
00ee68308b45bc5e2660cd833c3f81cc,3333,SAO PAULO,SP,Thiago Da Conceição,2019-01-02,2026-04-05T10:47:17.095Z,true,true,true,true,OK


Qtd linhas em tb_avaliacoes: 99224
Qtd linhas em tb_itens_pedidos: 112650
Qtd linhas em tb_produtos: 32951
Qtd linhas em tb_vendedores: 3095


In [0]:
# 1. Leitura das Tabelas Silver
# Foi feita releitura pra facilitar o criação das tabelas pro pipeline 
# Dessa forma, não é necessário rodar o notebook anterior sempre
df_avaliacoes = spark.table(f"{catalogo}.{schema_silver}.fat_avaliacoes_pedidos_validos")
df_itens_pedidos = spark.table(f"{catalogo}.{schema_silver}.fat_itens_pedidos_validos")
df_produtos = spark.table(f"{catalogo}.{schema_silver}.dim_produtos_validos")
df_vendedores = spark.table(f"{catalogo}.{schema_silver}.dim_vendedores_validos")

# 2. Transformação inicial simples pra evitar confusão de nomes ao buscar o estado do vendedor, informação que pode ser facilmente confundida ou gerar erros devido ao nome
df_vendedores = df_vendedores.withColumnRenamed("estado", "estado_vendedor")

# 3. Começando os joins das tabelas 
# O primeiro join a ser realizado será entre as tabelas de avaliação e itens_pedidos usando como chave o id_pedido
df_join = (
    df_avaliacoes
    .join(df_itens_pedidos, on="id_pedido", how="inner")
    .join(df_produtos, on="id_produto", how="inner")
    .join(df_vendedores, on="id_vendedor", how="inner")
)

# 4. Agrupamento e agregação
# O agg é a abreviação pra função de agregação 
# É isso que permite adicionar colunas advindas de cálculos, seja ele um count ou outra coisa mais complexa
df_agrupado = df_join.groupBy("categoria_produto", "nome_vendedor", "estado_vendedor").agg(
    F.count("id_avaliacao").alias("total_avaliacoes"),
    F.round(F.avg("nota_avaliacao"), 2).alias("avaliacao_media"),
    F.sum(F.when(F.col("nota_avaliacao") >= 4, 1).otherwise(0)).alias("total_avaliacoes_positivas"),
    F.sum(F.when(F.col("nota_avaliacao") <= 2, 1).otherwise(0)).alias("total_avaliacoes_negativas")
)

# 5. Cálculos adicionais
# Nessa etapa foram adicionados alguns cálculos a fim de facilitar a análise das informações necessárias 
df_avaliacoes_clientes = (
    df_agrupado
    .withColumn("percentual_satisfacao", F.round((F.col("total_avaliacoes_positivas") / F.col("total_avaliacoes")) * 100, 2))
    .select(
        "categoria_produto",
        "nome_vendedor",
        "estado_vendedor",
        "total_avaliacoes",
        "avaliacao_media",
        "total_avaliacoes_positivas",
        "total_avaliacoes_negativas",
        "percentual_satisfacao"
    )
)

# 6. Analisando se o dataframe possui as informações que deveria
df_avaliacoes_clientes.display()

# 7. Escrevendo a tabela na camada gold 
df_avaliacoes_clientes.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{schema_gold}.fat_avaliacoes_clientes")


categoria_produto,nome_vendedor,estado_vendedor,total_avaliacoes,avaliacao_media,total_avaliacoes_positivas,total_avaliacoes_negativas,percentual_satisfacao
MALAS_ACESSORIOS,Lucca Farias,SP,22,4.05,17,4,77.27
ALIMENTOS,Valentina Nascimento,SP,115,3.97,87,17,75.65
INFORMATICA_ACESSORIOS,Srta. Isabella Fonseca,MG,266,3.68,180,62,67.67
MOVEIS_DECORACAO,João Camargo,SP,146,3.75,95,32,65.07
PAPELARIA,Guilherme Cardoso,RS,11,4.91,11,0,100.0
MOVEIS_DECORACAO,Luigi Carvalho,RJ,59,4.42,51,4,86.44
BRINQUEDOS,Isabela Mendes,RJ,370,4.19,299,47,80.81
RELOGIOS_PRESENTES,Murilo Costa,SP,994,4.11,770,136,77.46
BEBES,Srta. Joana Da Cruz,RS,69,4.03,50,11,72.46
ESPORTE_LAZER,Sr. Diogo Moura,MG,50,4.24,42,3,84.0


### Exibição via Display()

In [0]:
# 1. PREPARAÇÃO: Agrupando as avaliações por Produto e por Vendedor

# Agrupamento de Produtos
df_rank_produtos = df_join.groupBy("nome_produto").agg(
    F.round(F.avg("nota_avaliacao"), 2).alias("avaliacao_media"),
    F.count("id_avaliacao").alias("total_avaliacoes")
)

# Agrupamento de Vendedores
df_rank_vendedores = df_join.groupBy("nome_vendedor").agg(
    F.round(F.avg("nota_avaliacao"), 2).alias("avaliacao_media"),
    F.count("id_avaliacao").alias("total_avaliacoes")
)

# 2. CRIAÇÃO DOS RANKINGS (Aplicando Ordenação e Desempate)

# 2.1. O Produto MAIS bem avaliado
df_produto_mais_avaliado = df_rank_produtos.orderBy(
    F.col("avaliacao_media").desc(),
    F.col("total_avaliacoes").desc()
)

# 2.2. O Produto MENOS bem avaliado
df_produto_menos_avaliado = df_rank_produtos.orderBy(
    F.col("avaliacao_media").asc(),
    F.col("total_avaliacoes").desc()
)

# 2.3. O Vendedor MAIS bem avaliado
df_vendedor_mais_avaliado = df_rank_vendedores.orderBy(
    F.col("avaliacao_media").desc(),
    F.col("total_avaliacoes").desc()
)

# 2.4. O Vendedor MENOS bem avaliado
df_vendedor_menos_avaliado = df_rank_vendedores.orderBy(
    F.col("avaliacao_media").asc(),
    F.col("total_avaliacoes").desc()
)

# 3. EXIBIÇÃO EM TELA
print("🏆 1. PRODUTO MAIS BEM AVALIADO:")
display(df_produto_mais_avaliado.limit(1))

print("⚠️ 2. PRODUTO MENOS BEM AVALIADO:")
display(df_produto_menos_avaliado.limit(1))

print("🏆 3. VENDEDOR MAIS BEM AVALIADO:")
display(df_vendedor_mais_avaliado.limit(1))

print("⚠️ 4. VENDEDOR MENOS BEM AVALIADO:")
display(df_vendedor_menos_avaliado.limit(1))

🏆 1. PRODUTO MAIS BEM AVALIADO:


nome_produto,avaliacao_media,total_avaliacoes
Caneta Esferográfica Avançado,5.0,18


⚠️ 2. PRODUTO MENOS BEM AVALIADO:


nome_produto,avaliacao_media,total_avaliacoes
Conjunto De Pincéis Ultra,1.0,7


🏆 3. VENDEDOR MAIS BEM AVALIADO:


nome_vendedor,avaliacao_media,total_avaliacoes
Luiz Otávio Abreu,5.0,34


⚠️ 4. VENDEDOR MENOS BEM AVALIADO:


nome_vendedor,avaliacao_media,total_avaliacoes
Sra. Fernanda Santos,1.0,8
